In [235]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter
import copy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import cross_val_score, KFold, cross_validate
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")


%matplotlib inline

# PREPROCESSING FOR TRAIN SET

In [236]:
# Split datasets

def split_sets(X, test_size=0.2, random_state=42):
    y = X['Survival Prediction']
    X = X.drop(columns=['Survival Prediction'])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    y_train.replace({'No': 0, 'Yes': 1})
    y_test.replace({'No': 0, 'Yes': 1})
    print(f"Dimension of X_train: {X_train.shape}")
    print(f"Dimension of X_test: {X_test.shape}")
    print(f"Dimension of y_train: {y_train.shape}")
    print(f"Dimension of y_test: {y_test.shape}")
    return X_train, X_test, y_train, y_test

In [237]:
# Data imputation for categorical variables (with the mode)

# Columns to process here: 'Healthcare Access' and 'Gender'

def df_cat_imputation(df, values_to_imput_cat):
  df_new = df.copy()
  mode_train={}

  for column in values_to_imput_cat.keys():
    mode_train[column] = df_new[df_new[column] != values_to_imput_cat[column]].loc[:, column].mode()[0]
    df_new.loc[df_new[column] == values_to_imput_cat[column], column] = mode_train[column]

  return df_new, mode_train

In [238]:
# Remove columns

# According to EDA, columns to remove: Transfusion History, Marital Status, Non Smoker (due to high correlation with 'Smoking History')

def remove_columns(df, columns_to_delete):
  for column in columns_to_delete:
    df = df.drop(columns=[column], errors='ignore')
  return df

In [239]:
# Remove duplicates and rows with null values

def remove_rows(df, y_train):
    # 1. Remove duplicates
    df_no_dups = df.drop_duplicates()
    duplicates_removed = len(df) - len(df_no_dups)
    #print(f"# duplicates removed: {duplicates_removed}")

    # 2. Remove rows with missing values
    df_clean = df_no_dups.dropna()
    rows_removed = len(df_no_dups) - len(df_clean)
    #print(f"Number of rows deleted for having missing values {rows_removed}")

    # Save indices from the clean df
    surviving_indices = df_clean.index

    # Filter y_train with surviving indices
    y_clean = y_train.loc[surviving_indices]

    # Reset indices if needed
    df_clean = df_clean.reset_index(drop=False)
    y_clean = y_clean.reset_index(drop=True)

    return df_clean, y_clean

In [240]:
# Standardize values in 'Urban or Real' column

def standardize_urbal_rural (df):
  df_new = df.copy()
  df_new['Urban or Rural'] = df['Urban or Rural'].str.lower()
  df_new['Urban or Rural'] = df['Urban or Rural'].str.capitalize()

  return df_new

In [241]:
# Encode nominal variables into booleans

# Nominal columns (Yes/No): 'Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease',
#        'Survival Prediction', 'Diabetes', 'Alcohol Consumption', 'Early Detection',
#        'Family History', 'Genetic Mutation'

def convert_into_bool(df, binary_cols):
  df_new = df.copy()
  for col in binary_cols:
        if col in df_new.columns:
            df_new[col] = df_new[col].map({'Yes': 1, 'No': 0}).astype(bool)

  return df_new

In [242]:
# Transform 'Date of Birth' column into 'Age' column

def create_age_column(df, reference_date='2025-01-01'):

      df_new = df.copy()
      if 'Date of Birth' in df_new.columns:
        # Create 'Age' Column
        try:
            df_new['Date of Birth'] = pd.to_datetime(df_new['Date of Birth'], errors='coerce')
            ref_date = pd.Timestamp(reference_date)
            df_new['Age'] = ((ref_date - df_new['Date of Birth']).dt.days / 365.25).round()
            df_new['Age'] = df_new['Age'].astype('float')

            # Delete 'Date of birth'
            df_new = df_new.drop(columns=['Date of Birth'])
            #print(f"'Date of Birth' transformed into 'Age'")
        except Exception as e:
            print(f"There was an error {e}")
      return df_new

In [243]:
# Preprocessing for numerical variables (winsorization for outliers, imputation with median)

# Numerical columns to preprocess here: 'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
#        'Tumor Size (mm)'

# Column 'Healthcare Costs' has negative values. These will be imputated with the median.

def df_num_imputation(df, numeric_cols):
  df_new = df.copy()
  stats_pre = {}

# Transform columns into float type
  for col in numeric_cols:
    if col in df_new.columns:
      df_new[col] = pd.to_numeric(df_new[col], errors='coerce')

# Remove outliers using 'Winsorization'
  for column in numeric_cols:
      Q1 = df_new[column].quantile(0.25)
      Q3 = df_new[column].quantile(0.75)
      IQR = Q3 - Q1

      lower_bound = Q1 - 1.5 * IQR
      upper_bound = Q3 + 1.5 * IQR

      # Clip outliers with upper or lower bound
      df_new[column] = df_new[column].clip(lower=lower_bound, upper=upper_bound)

      # Imputing negative values with the median
      median_c = df_new[column].median()
      df_new.loc[df_new[column] < 0, column] = median_c
      stats_pre[column] = {
          'lower_bound': lower_bound,
          'upper_bound': upper_bound,
          'median': median_c
      }

  # Returning the df and the stats_pre dictionary which will be used later to preprocess the test set
  return df_new, stats_pre

In [244]:
# Create dummy columns for categorical ordinal variables

# Those categoricals with less than 2 unique values will be label encoded.

# The main idea is to get rid of categorical columns by turning them into dummies

# Columns to process here
#'Cancer Stage', 'Country', 'Diet Risk', 'Gender', 'Healthcare Access',
#        'Insurance Costs', 'Insurance Status', 'Obesity BMI', 'Physical Activity',
#        'Screening History', 'Smoking History', 'Treatment Type', 'Urban or Rural'

def create_dummies(df, categorical_cols):
  df_new = df.copy()
  for col in categorical_cols:
    if col in categorical_cols:
      unique_values = df_new[col].nunique()
      if unique_values > 2:  # One-hot encoding will be performed on variables with more than 2 unique values
        dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=False)
        df_new = pd.concat([df_new, dummies], axis=1)
        df_new = df_new.drop(columns=[col])  # Drop original column after getting dummies

  # Identify remaining categorical columns

  non_numeric_cols = []
  for col in df_new.columns:
      if df_new[col].dtype == 'object':
          #print(f"Non numeric column found: {col}")
          non_numeric_cols.append(col)

  # Transform previous columns

  for col in non_numeric_cols:
    # Customized code for 'Gender' column
    if col in non_numeric_cols:
      if col == 'Gender' and 'Gender' in non_numeric_cols:
        df_new['Gender'] = df_new['Gender'].map({'M': 0, 'F': 1}).astype(bool)
      else:
        dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=True).astype(bool)
        df_new = pd.concat([df_new, dummies], axis=1)
        df_new = df_new.drop(columns=[col])

  # Check for non-numeric remaining columns

  for col in df_new.columns:
    if df_new[col].dtype == 'object':
      print(f"Error: Column {col} remains as non-numeric")

  return df_new

In [245]:
# Macro function for gathering the previous ones

def preprocess_cancer_df(X, y, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols, categorical_cols):

  X_clean = remove_columns(X, columns_to_delete) # Goes first because there are columns with empty values
  X_clean, mode_train = df_cat_imputation(X_clean, values_to_imput_cat) # Goes second because rows with values to be imputed are removed afterwards
  X_clean, y_processed = remove_rows(X_clean, y)
  X_clean = standardize_urbal_rural(X_clean)
  X_clean = convert_into_bool(X_clean, binary_cols)
  X_clean = create_age_column(X_clean)
  X_clean, stats_pre = df_num_imputation(X_clean, numeric_cols)
  X_processed = create_dummies(X_clean, categorical_cols)
  X_processed.reset_index(drop=True, inplace=True)
  X_processed.drop(columns='ID', inplace=True)

  return X_processed, y_processed, mode_train, stats_pre

In [252]:
def balance_features(X_train, y_train, columns_to_balance, random_state=42):

    # Reset indices to ensure alignment
    X_result = X_train.copy().reset_index(drop=True)
    y_result = y_train.copy().reset_index(drop=True)

    for col in columns_to_balance:
        print(f"\nBalancing column: {col}")
        print(f"Original distribution: {X_result[col].value_counts().to_dict()}")

        # Apply SMOTE
        smote = SMOTE(random_state=random_state)
        features = X_result.drop(columns=col)
        target = X_result[col]

        # Resampling
        features_balanced, target_balanced = smote.fit_resample(features, target)

        # Calculate how many synthetic samples were generated
        n_original = len(X_result)
        n_balanced = len(features_balanced)
        n_synthetic = n_balanced - n_original

        # Rebuild X_balanced
        X_balanced = pd.DataFrame(features_balanced, columns=features.columns)
        X_balanced[col] = target_balanced
        X_balanced = X_balanced.reset_index(drop=True)

        # Generate new values for y_balanced
        if n_synthetic > 0:
            # Generate synthetic values following the original distribution
            if isinstance(y_result, pd.Series):
                class_distribution = y_result.value_counts(normalize=True)
                classes = class_distribution.index.tolist()
                probs = class_distribution.values

                y_synthetic = pd.Series(
                    np.random.choice(classes, size=n_synthetic, p=probs),
                    name=y_result.name
                )

                y_balanced = pd.concat([y_result, y_synthetic]).reset_index(drop=True)

            else:  # DataFrame
                y_synthetic = pd.DataFrame(index=range(n_synthetic))

                for y_col in y_result.columns:
                    class_distribution = y_result[y_col].value_counts(normalize=True)
                    classes = class_distribution.index.tolist()
                    probs = class_distribution.values

                    y_synthetic[y_col] = np.random.choice(classes, size=n_synthetic, p=probs)

                y_balanced = pd.concat([y_result, y_synthetic]).reset_index(drop=True)

        else:
            y_balanced = y_result.copy()

        # Update for next iteration
        X_result = X_balanced
        y_result = y_balanced

        print(f"Distribution after SMOTE: {X_result[col].value_counts().to_dict()}")
        print(f"Dimensions: X={X_result.shape}, y={y_result.shape}")

    return X_result, y_result

In [246]:
columns_to_delete = ['Transfusion History', 'Marital Status', 'Non Smoker']

values_to_imput_cat = {
        'Healthcare Access': '?',
        'Gender': 'P'
    }

binary_cols = [
        'Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease',
        'Survival Prediction', 'Diabetes', 'Alcohol Consumption', 'Early Detection',
        'Family History', 'Genetic Mutation'
    ]

numeric_cols = [
        'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
        'Tumor Size (mm)'
    ]

categorical_cols = [
        'Cancer Stage', 'Country', 'Diet Risk', 'Gender', 'Healthcare Access',
        'Insurance Costs', 'Insurance Status', 'Obesity BMI', 'Physical Activity',
        'Screening History', 'Smoking History', 'Treatment Type', 'Urban or Rural'
]

columns_to_balance = ['Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease']

# PREPROCESSING FOR TEST SET

In [ ]:
def testdf_cat_imputation(df, mode_train):

  df_new = df.copy()

  for column in values_to_imput_cat.keys():
    df_new.loc[df_new[column] == values_to_imput_cat[column], column] = mode_train[column]

  return df_new, mode_train

#---

In [248]:
cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/4a73ca4928f2b95f960cd9b9f44c4700244ed553/data/raw/patient_train_data.csv',
                        encoding='UTF-8',
                        index_col=0,
                        sep=',',
                        on_bad_lines='skip',
                        quoting=3)

In [249]:
X_train, X_test, y_train, y_test = split_sets(cancer_df)

Dimension of X_train: (60028, 30)
Dimension of X_test: (15007, 30)
Dimension of y_train: (60028,)
Dimension of y_test: (15007,)


In [250]:
X_processed, y_processed, mode_train, stats_pre = preprocess_cancer_df(X_train, y_train, columns_to_delete, values_to_imput_cat, binary_cols, numeric_cols, categorical_cols)

In [253]:
X_train_balanced, y_train_balanced = balance_features(X_processed, y_processed, columns_to_balance)


Balancing column: Diabetes History
Original distribution: {False: 59128, True: 8}
Distribution after SMOTE: {False: 59128, True: 59128}
Dimensions: X=(118256, 59), y=(118256,)

Balancing column: Heart Disease History
Original distribution: {False: 112273, True: 5983}
Distribution after SMOTE: {False: 112273, True: 112273}
Dimensions: X=(224546, 59), y=(224546,)

Balancing column: Inflammatory Bowel Disease
Original distribution: {False: 198518, True: 26028}
Distribution after SMOTE: {False: 198518, True: 198518}
Dimensions: X=(397036, 59), y=(397036,)
